In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.table import Table
from researchcodes import (
    define_column_desc, 
    iter_multi_csv_chunks, 
    write_std_h5, 
)

# Process the text catalog

The RA and Dec errors need calculation, so it's better to process ahead of reading and writing.

In the meantime, we can drop unnecessary columns.

In [2]:
tbl = Table.read("landolt2009table.fit", hdu=1)
raw_text_catalog = tbl.to_pandas()
raw_text_catalog = raw_text_catalog.rename(
    columns={
        "Name": "id_name",
        "Vmag": "Bessel_V",
        "e_Vmag": "Bessel_V_err",
        "RAJ2000": "ra",
        "DEJ2000": "dec",
    }
)
raw_text_catalog

,id_name,Bessel_V,Bessel_V_err,B-V,e_B-V,U-B,e_U-B,V-R,e_V-R,R-I,e_R-I,V-I,e_V-I,Nobs,Nnig,ra,dec,recno
0,b'TPhe I',14.820,0.0026,0.764,0.0032,0.338,0.0072,0.422,0.0036,0.395,0.0098,0.817,0.0110,25,13,7.519137,-46.469492,1
1,b'TPhe A',14.651,0.0028,0.793,0.0046,0.380,0.0071,0.435,0.0019,0.405,0.0035,0.841,0.0032,29,12,7.539975,-46.524697,2
2,b'TPhe H',14.942,0.0029,0.740,0.0029,0.225,0.0071,0.425,0.0035,0.425,0.0077,0.851,0.0098,23,12,7.540346,-46.456750,3
3,b'TPhe B',12.334,0.0115,0.405,0.0026,0.156,0.0039,0.262,0.0020,0.271,0.0019,0.535,0.0035,29,17,7.567971,-46.466269,4
4,b'TPhe C',14.376,0.0022,-0.298,0.0024,-1.217,0.0043,-0.148,0.0038,-0.211,0.0133,-0.360,0.0149,39,23,7.570750,-46.539278,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590,b'115 420',11.160,0.0006,0.467,0.0005,-0.019,0.0008,0.288,0.0005,0.293,0.0006,0.581,0.0007,91,82,355.652004,1.099672,591
591,b'115 271',9.693,0.0005,0.612,0.0004,0.109,0.0008,0.354,0.0003,0.349,0.0005,0.702,0.0007,118,96,355.674271,0.753650,592
592,b'115 516',10.431,0.0006,1.028,0.0005,0.760,0.0012,0.564,0.0004,0.534,0.0004,1.099,0.0006,100,89,356.064062,1.236822,593
593,b'BD +1 4774',8.993,0.0012,1.434,0.0016,1.105,0.0020,0.964,0.0008,1.081,0.0016,2.047,0.0016,6,6,357.302192,2.401228,594


In [3]:
raw_text_catalog["Bessel_B"] = raw_text_catalog["Bessel_V"] + raw_text_catalog["B-V"]
raw_text_catalog["Bessel_U"] = raw_text_catalog["Bessel_V"] + raw_text_catalog["B-V"] + raw_text_catalog["U-B"]
raw_text_catalog["Bessel_R"] = raw_text_catalog["Bessel_V"] - raw_text_catalog["V-R"]
raw_text_catalog["Bessel_I"] = raw_text_catalog["Bessel_V"] - raw_text_catalog["V-I"]

raw_text_catalog["Bessel_B_err"] = np.sqrt(
    raw_text_catalog["Bessel_V_err"]**2 + raw_text_catalog["e_B-V"]**2
)

raw_text_catalog["Bessel_U_err"] = np.sqrt(
    raw_text_catalog["Bessel_V_err"]**2 +
    raw_text_catalog["e_B-V"]**2 +
    raw_text_catalog["e_U-B"]**2
)

raw_text_catalog["Bessel_R_err"] = np.sqrt(
    raw_text_catalog["Bessel_V_err"]**2 + raw_text_catalog["e_V-R"]**2
)

raw_text_catalog["Bessel_I_err"] = np.sqrt(
    raw_text_catalog["Bessel_V_err"]**2 + raw_text_catalog["e_V-I"]**2
)

In [4]:
raw_text_catalog

,id_name,Bessel_V,Bessel_V_err,B-V,e_B-V,U-B,e_U-B,V-R,e_V-R,R-I,...,dec,recno,Bessel_B,Bessel_U,Bessel_R,Bessel_I,Bessel_B_err,Bessel_U_err,Bessel_R_err,Bessel_I_err
0,b'TPhe I',14.820,0.0026,0.764,0.0032,0.338,0.0072,0.422,0.0036,0.395,...,-46.469492,1,15.584000,15.922000,14.398,14.002999,0.004123,0.008297,0.004441,0.011303
1,b'TPhe A',14.651,0.0028,0.793,0.0046,0.380,0.0071,0.435,0.0019,0.405,...,-46.524697,2,15.444000,15.824000,14.216,13.810000,0.005385,0.008911,0.003384,0.004252
2,b'TPhe H',14.942,0.0029,0.740,0.0029,0.225,0.0071,0.425,0.0035,0.425,...,-46.456750,3,15.682000,15.907001,14.517,14.091001,0.004101,0.008199,0.004545,0.010220
3,b'TPhe B',12.334,0.0115,0.405,0.0026,0.156,0.0039,0.262,0.0020,0.271,...,-46.466269,4,12.738999,12.895000,12.072,11.799000,0.011790,0.012419,0.011673,0.012021
4,b'TPhe C',14.376,0.0022,-0.298,0.0024,-1.217,0.0043,-0.148,0.0038,-0.211,...,-46.539278,5,14.078000,12.861000,14.524,14.736000,0.003256,0.005394,0.004391,0.015062
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590,b'115 420',11.160,0.0006,0.467,0.0005,-0.019,0.0008,0.288,0.0005,0.293,...,1.099672,591,11.627000,11.608000,10.872,10.579000,0.000781,0.001118,0.000781,0.000922
591,b'115 271',9.693,0.0005,0.612,0.0004,0.109,0.0008,0.354,0.0003,0.349,...,0.753650,592,10.305000,10.414001,9.339,8.991000,0.000640,0.001025,0.000583,0.000860
592,b'115 516',10.431,0.0006,1.028,0.0005,0.760,0.0012,0.564,0.0004,0.534,...,1.236822,593,11.459000,12.219000,9.867,9.332000,0.000781,0.001432,0.000721,0.000849
593,b'BD +1 4774',8.993,0.0012,1.434,0.0016,1.105,0.0020,0.964,0.0008,1.081,...,2.401228,594,10.427000,11.532000,8.029,6.946000,0.002000,0.002828,0.001442,0.002000


In [5]:
# drop the columns we don't want!

process_text_catalog = raw_text_catalog.drop(
    columns=[
        "Nobs", "Nnig", "recno",
        "B-V", "U-B", "V-R", "R-I", "V-I",
        "e_B-V", "e_U-B", "e_V-R", "e_R-I", "e_V-I",

    ]
)

process_text_catalog                   

,id_name,Bessel_V,Bessel_V_err,ra,dec,Bessel_B,Bessel_U,Bessel_R,Bessel_I,Bessel_B_err,Bessel_U_err,Bessel_R_err,Bessel_I_err
0,b'TPhe I',14.820,0.0026,7.519137,-46.469492,15.584000,15.922000,14.398,14.002999,0.004123,0.008297,0.004441,0.011303
1,b'TPhe A',14.651,0.0028,7.539975,-46.524697,15.444000,15.824000,14.216,13.810000,0.005385,0.008911,0.003384,0.004252
2,b'TPhe H',14.942,0.0029,7.540346,-46.456750,15.682000,15.907001,14.517,14.091001,0.004101,0.008199,0.004545,0.010220
3,b'TPhe B',12.334,0.0115,7.567971,-46.466269,12.738999,12.895000,12.072,11.799000,0.011790,0.012419,0.011673,0.012021
4,b'TPhe C',14.376,0.0022,7.570750,-46.539278,14.078000,12.861000,14.524,14.736000,0.003256,0.005394,0.004391,0.015062
...,...,...,...,...,...,...,...,...,...,...,...,...,...
590,b'115 420',11.160,0.0006,355.652004,1.099672,11.627000,11.608000,10.872,10.579000,0.000781,0.001118,0.000781,0.000922
591,b'115 271',9.693,0.0005,355.674271,0.753650,10.305000,10.414001,9.339,8.991000,0.000640,0.001025,0.000583,0.000860
592,b'115 516',10.431,0.0006,356.064062,1.236822,11.459000,12.219000,9.867,9.332000,0.000781,0.001432,0.000721,0.000849
593,b'BD +1 4774',8.993,0.0012,357.302192,2.401228,10.427000,11.532000,8.029,6.946000,0.002000,0.002828,0.001442,0.002000


In [6]:
# save the process catalog text file

process_text_catalog.to_csv(
    "ESO_Landolt_Equatorial_Standards.dat",
    sep=",",
    index=False,
)

# Define the HFD5 column description

In [7]:
# Define magnitude column names

# Note this magnitude column order is not necessarily
# to be the same as the column order in the text file. 

# It is OK as long as the filter names in 
# `magnitude_column_names` matches `colnames` defined 
# when reading the text file

magnitude_column_names = [
     "Bessel_U","Bessel_B","Bessel_V","Bessel_R","Bessel_I",
    "Bessel_U_err","Bessel_B_err","Bessel_V_err","Bessel_R_err","Bessel_I_err",
]

# define h5 file column description
h5_columns = define_column_desc(
    magnitude_column_names=magnitude_column_names, 
    id_name_length=20, 
)

In [8]:
h5_columns

{'id_name': StringCol(itemsize=20, shape=(), dflt=np.bytes_(b''), pos=0),
 'ra': Float32Col(shape=(), dflt=np.float32(0.0), pos=1),
 'ra_err': Float32Col(shape=(), dflt=np.float32(0.0), pos=2),
 'dec': Float32Col(shape=(), dflt=np.float32(0.0), pos=3),
 'dec_err': Float32Col(shape=(), dflt=np.float32(0.0), pos=4),
 'Bessel_U': Float32Col(shape=(), dflt=np.float32(0.0), pos=5),
 'Bessel_B': Float32Col(shape=(), dflt=np.float32(0.0), pos=6),
 'Bessel_V': Float32Col(shape=(), dflt=np.float32(0.0), pos=7),
 'Bessel_R': Float32Col(shape=(), dflt=np.float32(0.0), pos=8),
 'Bessel_I': Float32Col(shape=(), dflt=np.float32(0.0), pos=9),
 'Bessel_U_err': Float32Col(shape=(), dflt=np.float32(0.0), pos=10),
 'Bessel_B_err': Float32Col(shape=(), dflt=np.float32(0.0), pos=11),
 'Bessel_V_err': Float32Col(shape=(), dflt=np.float32(0.0), pos=12),
 'Bessel_R_err': Float32Col(shape=(), dflt=np.float32(0.0), pos=13),
 'Bessel_I_err': Float32Col(shape=(), dflt=np.float32(0.0), pos=14),
 'ipix': Int32Col(s

# Read the cvs files

In [9]:
# file path
root_dir = Path(".")
file = root_dir / "ESO_Landolt_Equatorial_Standards.dat"

In [10]:
# csv column names
# here the columns names must match the h5_columns
colnames = process_text_catalog.columns.to_list()

dataframe_iterator = iter_multi_csv_chunks(
    files=file, 
    chunksize=100, 
    read_csv_kwargs={
        "sep": ",", 
        "engine": "python", 
        "header": None, 
        "names": colnames,
        "skiprows": 1, 
    }
)

In [11]:
table_attrs = {
    "source": "https://www2.keck.hawaii.edu/inst/common/landolt_stds.html",
    "ra_unit": "deg",
    "dec_unit": "deg",
    "ra_err_unit": "arcsec",
    "dec_err_unit": "arcsec",
    "version": "Landolt 2009",
    "mag_system": {
        "Bessel_U": "Johnson–Kron–Cousins",
        "Bessel_B": "Johnson–Kron–Cousins",
        "Bessel_V": "Johnson–Kron–Cousins",
        "Bessel_R": "Johnson–Kron–Cousins",
        "Bessel_I": "Johnson–Kron–Cousins",
    },
}

In [12]:
write_std_h5(
    dataframe_iterator=dataframe_iterator, 
    ra_dec_hmsdms=False,
    h5_output_path=Path("ESO_Landolt_Equatorial_Standards.h5",), 
    group_where="/ESO", 
    group_name="landolt", 
    group_title="Landolt Equatorial Standards", 
    table_name="std", 
    table_description=h5_columns, 
    table_title="Standard Stars", 
    table_attrs=table_attrs, 
    nside=512, 
    bucket_size=1536 , 
)

Files:   0%|          | 0/1 [00:00<?, ?it/s]

Chunks:   0%|                                             | 0/6 [00:00<?, ?it/s]